In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install necessary libraries
!pip install datasets requests --quiet

# Import required libraries for data processing and ML
import pandas as pd
import numpy as np
import requests
import json
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
import pickle
import os
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, log_loss
import matplotlib.pyplot as plt

!pip install gensim --upgrade --quiet
from gensim.models import Word2Vec

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier


# Login to Hugging Face Hub
from huggingface_hub import login
login(token="your_token")

# Load dataset from Hugging Face
from datasets import load_dataset
dataset = load_dataset("sapienzanlp/nlp2025_hw1_cultural_dataset")
df_train = dataset["train"].to_pandas()
df_dev = dataset["validation"].to_pandas()


# TRAINING

In [ ]:
# Preprocessing: Combine text columns for training and validation
def clean_text(text):
   words = text.split()
   return ' '.join([word for word in words if word.lower() not in ENGLISH_STOP_WORDS])


In [ ]:
# Preprocessing: Combine text columns for training and validation

X_train_text = (df_train["name"] + " " + df_train["description"] + " " + df_train["type"] + " " + df_train["category"]).apply(clean_text).tolist()
y_train = df_train["label"].tolist()

X_dev_text = (df_dev["name"] + " " + df_dev["description"] + " " + df_dev["type"] + " " + df_dev["category"]).apply(clean_text).tolist()
y_dev = df_dev["label"].tolist()



In [ ]:
# TF-IDF Vectorization: Convert text data into numerical features
vectorizer = TfidfVectorizer(max_features=5000)  # Initialize the vectorizer with a max of 5000 features
X_train_tfidf = vectorizer.fit_transform(X_train_text)  # Fit and transform the training data
X_dev_tfidf = vectorizer.transform(X_dev_text)  # Transform the validation data


In [ ]:
# Word2Vec Embeddings
sentences = [text.split() for text in X_train_text]
word2vec_model = Word2Vec(sentences, vector_size=50, window=5, min_count=1, workers=4)

def transform_text(text):
   words = text.split()
   return np.mean([word2vec_model.wv[word] for word in words if word in word2vec_model.wv], axis=0)

X_train_w2v = np.array([transform_text(text) for text in X_train_text])
X_dev_w2v = np.array([transform_text(text) for text in X_dev_text])


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_w2v_scaled = scaler.fit_transform(X_train_w2v)
X_dev_w2v_scaled = scaler.transform(X_dev_w2v)


In [ ]:
# Proprietà semantiche utili da Wikidata
properties_of_interest = {
	"P31": "instance of",
	"P279": "subclass of",
	"P106": "occupation",
	"P136": "genre",
	"P101": "field of work",
	"P170": "creator",
	"P17": "country",
	"P495": "country of origin",
	"P27": "citizenship"
}

# Function to extract "country of origin" from Wikidata with caching
wikidata_cache = {}  # Cache for storing fetched country data

def get_country_feature(wikidata_url):
    # Check if the country data is already cached
    if wikidata_url in wikidata_cache:
        return wikidata_cache[wikidata_url]

    try:
        # Extract the entity ID from the Wikidata URL
        entity_id = wikidata_url.split("/")[-1]
        url = f"https://www.wikidata.org/wiki/Special:EntityData/{entity_id}.json"

        # Fetch data from Wikidata
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            claims = data["entities"][entity_id]["claims"]

            # Look for country-related properties
            extracted_ids = []
            for prop in properties_of_interest.keys():
                if prop in claims:
                    for claim in claims[prop]:
                        try:
                            value = claim["mainsnak"]["datavalue"]["value"]
                            if isinstance(value, dict) and "id" in value:
                                extracted_ids.append(f"{prop}:{value['id']}")
                        except:
                            continue


            # Cache the result and add a small delay to avoid rate limiting
            wikidata_cache[wikidata_url] =  extracted_ids

            time.sleep(0.2)
            return extracted_ids

    except:
        pass

    # Default to agnostic if there's an error or no data
    wikidata_cache[wikidata_url] = [1, 0, 0]
    return [1, 0, 0]


In [ ]:
# Extract Wikidata features for training and validation data
X_train_wiki = [get_country_feature(url) for url in df_train["item"]]  # Extract features for training URLs
X_dev_wiki = [get_country_feature(url) for url in df_dev["item"]]  # Extract features for validation URLs

# Ensure all elements in X_train_wiki and X_dev_wiki are strings:
X_train_wiki = [[str(item) for item in sublist] for sublist in X_train_wiki]
X_dev_wiki = [[str(item) for item in sublist] for sublist in X_dev_wiki]


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer(sparse_output=True)
X_train_wiki = mlb.fit_transform(X_train_wiki)
X_dev_wiki = mlb.transform(X_dev_wiki)

In [ ]:
# Combine TF-IDF, Word2Vec, and Wikidata features for training and validation data
X_train_combined = hstack([X_train_tfidf, X_train_w2v, X_train_wiki]) # Combine TF-IDF, Word2Vec, and Wikidata features for training
X_dev_combined = hstack([X_dev_tfidf, X_dev_w2v, X_dev_wiki]) # Combine TF-IDF, Word2Vec, and Wikidata features for validation


In [ ]:
# Balance the classes using SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_combined, y_train)

In [ ]:
# Train Logistic Regression model
lrm = LogisticRegressionCV(max_iter=1000, cv=5, scoring='accuracy', refit=True)
lrm.fit(X_train_balanced, y_train_balanced) # Train the model using the combined features and labels

In [ ]:
# Make predictions and evaluate the model
y_pred1 = lrm.predict(X_dev_combined)  # Predict labels for the validation set
print(classification_report(y_dev, y_pred1))  # Print classification report for model evaluation

In [ ]:
# Train Random Forest Model
from sklearn.ensemble import RandomForestClassifier
rfm = RandomForestClassifier(n_estimators=100, random_state=42)
rfm.fit(X_train_balanced, y_train_balanced)

In [ ]:
# Make predictions and evaluate the model
y_pred2 = rfm.predict(X_dev_combined)  # Predict labels for the validation set
print(classification_report(y_dev, y_pred2))  # Print classification report for model evaluation

In [ ]:
# Train SVM Model
from sklearn.svm import SVC
svm = SVC(kernel='linear')  # o 'rbf'
svm.fit(X_train_balanced, y_train_balanced)

In [ ]:
# Make predictions and evaluate the model
y_pred3 = svm.predict(X_dev_combined)  # Predict labels for the validation set
print(classification_report(y_dev, y_pred3))  # Print classification report for model evaluation

In [ ]:
# Choose model (we choose SVM because have the best accuracy)
y_pred = y_pred3
model = svm

In [ ]:
# Save the model and tokenizer (vectorizer)
import pickle
import os

os.makedirs("/content/drive/MyDrive/The_Giadas_shared_folder/final_model2", exist_ok=True)  # Create the directory for saving the model if it doesn't exist

# Save the TF-IDF vectorizer
with open("/content/drive/MyDrive/The_Giadas_shared_folder/final_model2/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

# Save the Logistic Regression model
with open("/content/drive/MyDrive/The_Giadas_shared_folder/final_model2/model.pkl", "wb") as f:
    pickle.dump(model, f)

print("\nModel and tokenizer saved successfully in final_model2!")


In [ ]:
from sklearn.metrics import classification_report, precision_recall_fscore_support
# Funzione per plottare le metriche
def plot_metrics(y_true, y_pred):
    precision, recall, fscore, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    accuracy = accuracy_score(y_true, y_pred)
    metrics = {'Precision': precision, 'Recall': recall, 'F1-Score': fscore, 'Accuracy': accuracy}

    plt.figure(figsize=(8, 5))
    plt.bar(metrics.keys(), metrics.values(), color=['blue', 'green', 'red', 'purple'])
    plt.ylim(0, 1)
    plt.title('Classification Metrics')
    plt.xlabel('Metrics')
    plt.ylabel('Score')
    plt.show()

# Plot delle metriche
plot_metrics(y_dev, y_pred)

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Calcola la matrice
cm = confusion_matrix(y_dev, y_pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)

# Mostra
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix")
plt.show()

import seaborn as sns

plt.figure(figsize=(8, 4))
sns.countplot(x=y_pred, order=sorted(set(y_pred)))
plt.title("Distribution of the Predictions per Classes")
plt.xlabel("Predicted classes")
plt.ylabel("Number of instances")
plt.xticks(rotation=45)
plt.show()

report = classification_report(y_dev, y_pred, output_dict=True)
df_report = pd.DataFrame(report).transpose()

plt.figure(figsize=(8, 6))
sns.heatmap(df_report.iloc[:-1, :-1], annot=True, cmap="YlGnBu")
plt.title("Classification Report - Heatmap")
plt.show()



# TEST

In [ ]:
# Proprietà semantiche utili da Wikidata
properties_of_interest = {
	"P31": "instance of",
	"P279": "subclass of",
	"P106": "occupation",
	"P136": "genre",
	"P101": "field of work",
	"P170": "creator",
	"P17": "country",
	"P495": "country of origin",
	"P27": "citizenship"
}

# Function to extract "country of origin" from Wikidata with caching
wikidata_cache = {}  # Cache for storing fetched country data

def get_country_feature(wikidata_url):
    # Check if the country data is already cached
    if wikidata_url in wikidata_cache:
        return wikidata_cache[wikidata_url]

    try:
        # Extract the entity ID from the Wikidata URL
        entity_id = wikidata_url.split("/")[-1]
        url = f"https://www.wikidata.org/wiki/Special:EntityData/{entity_id}.json"

        # Fetch data from Wikidata
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            claims = data["entities"][entity_id]["claims"]

            # Look for country-related properties
            extracted_ids = []
            for prop in properties_of_interest.keys():
                if prop in claims:
                    for claim in claims[prop]:
                        try:
                            value = claim["mainsnak"]["datavalue"]["value"]
                            if isinstance(value, dict) and "id" in value:
                                extracted_ids.append(f"{prop}:{value['id']}")
                        except:
                            continue


            # Cache the result and add a small delay to avoid rate limiting
            wikidata_cache[wikidata_url] =  extracted_ids

            time.sleep(0.2)
            return extracted_ids

    except:
        pass

    # Default to agnostic if there's an error or no data
    wikidata_cache[wikidata_url] = [1, 0, 0]
    return [1, 0, 0]


In [ ]:
# 📝 Preprocessing del test set
test_data = pd.read_csv('/content/drive/MyDrive/The_Giadas_shared_folder/test_unlabeled.csv')

# Crea il testo di input combinando item_name e description

X_test_text = (test_data["name"] + " " + test_data["description"] + " " + test_data["type"] + " " + test_data["category"]).apply(clean_text).tolist()

# 🚀 Carica il modello e il tokenizer TF-IDF
with open("/content/drive/MyDrive/The_Giadas_shared_folder/final_model2/vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

with open("/content/drive/MyDrive/The_Giadas_shared_folder/final_model2/model.pkl", "rb") as f:
    model = pickle.load(f)

# Trasforma il testo usando TF-IDF
X_test_vec = vectorizer.transform(X_test_text)

# Word2Vec Embeddings
X_test_w2v = np.array([transform_text(text) for text in X_test_text])

# Estrazione delle Wikidata features
X_test_wikidata = [get_country_feature(url) for url in test_data["item"]]
# Ensure all elements in X_test_wiki and X_dev_wiki are strings:
X_test_wikidata = [[str(item) for item in sublist] for sublist in X_test_wikidata]


from sklearn.preprocessing import MultiLabelBinarizer

# Assuming 'mlb' is the MultiLabelBinarizer fitted during training
X_test_wikidata = mlb.transform(X_test_wikidata)  # Use the same mlb object from training

# Combina TF-IDF + Wikidata features
from scipy.sparse import hstack

X_test_combined = hstack([X_test_vec, X_test_w2v, X_test_wikidata])


In [ ]:
# Predizione sul test set
y_test_pred = model.predict(X_test_combined)

In [ ]:
# Creazione del DataFrame finale
test_data["label"] = y_test_pred

In [ ]:
# Salvataggio del file CSV per la submission
test_data[["item", "name", "description", "type", "category", "label"]].to_csv("/content/drive/MyDrive/The_Giadas_shared_folder/the_Giadas_output_modello2.csv", index=False)

In [ ]:
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.countplot(x=y_test_pred, order=sorted(set(y_test_pred)))
plt.title("Distribution of the Predictions per Classes")
plt.xlabel("Predicted classes")
plt.ylabel("Number of instances")
plt.xticks(rotation=45)
plt.show()
